In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [2]:
df = sns.load_dataset('titanic')
df['age'] = df['age'].fillna(df['age'].median())
df['embarked'] = df['embarked'].fillna(df['embarked'].mode()[0])
df = df.drop(columns=['deck'])
df = pd.get_dummies(df, columns=['sex', 'embarked'], drop_first=True)

In [3]:
X = df[['pclass', 'age', 'sibsp', 'parch', 'fare', 'sex_male']]
y = df['survived']

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [5]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
predictions = model.predict(X_test)

In [6]:
print(classification_report(y_test, predictions))

              precision    recall  f1-score   support

           0       0.81      0.88      0.84       105
           1       0.80      0.72      0.76        74

    accuracy                           0.81       179
   macro avg       0.81      0.80      0.80       179
weighted avg       0.81      0.81      0.81       179



Accuracy akela misleading ho sakta hai jab dataset imbalanced ho — jaise agar zyada tar log "not survived" hon, to model sirf hamesha "not survived" keh kar bhi zyada accuracy paa sakta hai, jabke woh kabhi kisi ko sahi "survived" predict nahi kar raha. Mere model mein class 1 (survived) ka recall (0.72) class 0 (0.88) se kam hai — matlab model survivors ko miss kar raha hai zyada, lekin overall accuracy (0.81) yeh baat chhupa deti hai. Isliye precision, recall aur F1-score dekhna zaroori hai, sirf accuracy pe bharosa nahi karna chahiye.

In [7]:
param_grid = {'C': [0.01, 0.1, 1, 10], 'max_iter': [500, 1000, 2000]}
grid_search = GridSearchCV(LogisticRegression(), param_grid, cv=5, scoring='accuracy')
grid_search.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=LogisticRegression(),
             param_grid={'C': [0.01, 0.1, 1, 10],
                         'max_iter': [500, 1000, 2000]},
             scoring='accuracy')

In [8]:
print('Best parameters:', grid_search.best_params_)
best_model = grid_search.best_estimator_

Best parameters: {'C': 0.1, 'max_iter': 500}


In [9]:
tuned_predictions = best_model.predict(X_test)
tuned_accuracy = accuracy_score(y_test, tuned_predictions)
print('Original Accuracy:', 0.81)
print('Tuned Accuracy:', tuned_accuracy)

Original Accuracy: 0.81
Tuned Accuracy: 0.8212290502793296


## Before vs After Tuning

| Metric | Original Model | Tuned Model |
|---|---|---|
| Accuracy | 0.81 | 0.8212 |
| Parameters | default (C=1.0, max_iter=1000) | C=0.1, max_iter=500 |

Hyperparameter tuning se accuracy mein thoda improvement hua — 0.81 se 0.8212 tak, yani lagbhag 1%. Yeh dikhata hai ke default settings pehle se hi kaafi acha kaam kar rahi thin, lekin GridSearchCV ke zariye tuning ne model ko thoda aur behtar bana diya.